# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`
This notebook demonstrates how to load, examine, and process the FAIR^2 dataset using the `mlcroissant` library. We will follow a step-by-step approach modeled on the `mlcroissant` best practices and the notebook template provided.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the FAIR^2 dataset Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset title and description from metadata object
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id`. This step allows us to identify the structure of the dataset before loading all records.

In [ ]:
# List all available record sets and their IDs.
print("Available record sets:")
for record_set in dataset.record_sets:
    print(f"- Name: {record_set.name}, @id: {record_set.id}")

# Let's inspect the first record set in detail as an example.
if len(dataset.record_sets) == 0:
    print("No record sets found in this dataset.")
else:
    # For this specific FAIR^2 dataset, fetch the first record set.
    first_record_set = dataset.record_sets[0]
    print(f"\nFields in record set '{first_record_set.name}' (@id: {first_record_set.id}):")
    for field in first_record_set.fields:
        print(f"  - Field: {field.name}, @id: {field.id}, dataType: {field.data_type}")


### Sample Record Iteration by `@id`
Now let's print the first few records of the principal record set by `@id`. (If you want to see other record sets, change the `record_set_id` variable below.)

In [ ]:
# Set the record set @id (use the actual @id found from the previous cell)
if len(dataset.record_sets) > 0:
    record_set_id = dataset.record_sets[0].id
    print(f"Iterating records for record set @id: {record_set_id}")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(f"Record {i+1}: {record}")
        if i >= 2:
            print("... (showing only first 3 records)")
            break
else:
    print('No record sets were found; skipping this step.')

## 3. Data Extraction
Load data from each record set into a `pandas.DataFrame` for analysis. This block is dynamic and reads all record sets referenced by their `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records into a dataframe
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set @id: {record_set_id} with shape {df.shape}")

if len(record_set_ids) > 0:
    chosen_record_set_id = record_set_ids[0]
    print(f"\nColumns in dataframe for record set '{chosen_record_set_id}':")
    print(dataframes[chosen_record_set_id].columns.tolist())
    print("\nFirst five rows of the data:")
    display(dataframes[chosen_record_set_id].head())
else:
    print('No record sets/dataframes found.')

## 4. Exploratory Data Analysis (EDA)
In this section, we will:
- Select a **numeric field** (referenced by its `@id`) from the main record set.
- Filter records based on a numerical criterion.
- Normalize the selected numeric field.
- Optionally group the results by a categorical or grouping field (also by its `@id`).

In [ ]:
# Choose main record set and field @ids (replace with valid ids from your dataset if different)
if len(dataset.record_sets) > 0:
    record_set = dataset.record_sets[0]
    record_set_id = record_set.id
    df = dataframes[record_set_id]
    
    # Find a numeric field
    numeric_field_obj = None
    for field in record_set.fields:
        # Heuristic: choose a field with dataType 'Integer' or 'Float' and present in df columns
        if (field.data_type in ['schema:Integer', 'schema:Float', 'Integer', 'Float']) and field.id in df.columns:
            numeric_field_obj = field
            break
    
    if numeric_field_obj is not None:
        numeric_field_id = numeric_field_obj.id
        print(f"Using numeric field: {numeric_field_obj.name} (@id: {numeric_field_id})")

        # Choose a threshold (10 used here as a demonstration; use domain-specific value as needed)
        threshold = 10
        if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
            filtered_df = df[df[numeric_field_id] > threshold].copy()  # copy to avoid SettingWithCopyWarning
            print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} records")
            display(filtered_df.head())

            # Normalize the numeric field (Z-score normalization)
            norm_col = f"{numeric_field_id}_normalized"
            filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"\nNormalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, norm_col]].head())

            # Find a grouping field (categorical)
            group_field_obj = None
            for field in record_set.fields:
                # Heuristic: has dataType 'Text' or 'String', is not the numeric field, and present in columns
                if (field.data_type in ['schema:Text', 'Text', 'String', 'schema:String']) \
                    and field.id != numeric_field_id and field.id in filtered_df.columns:
                    group_field_obj = field
                    break
            if group_field_obj is not None:
                group_field_id = group_field_obj.id
                print(f"\nGrouping by: {group_field_obj.name} (@id: {group_field_id})")
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
                display(grouped_df.head())
            else:
                print("No suitable grouping field found for EDA.")
        else:
            print(f"Field {numeric_field_id} is not numeric in the DataFrame.")
    else:
        print('No numeric field was found for EDA.')
else:
    print('No record sets found in dataset; skipping EDA.')

## 5. Visualization
Let us visualize the distribution of the chosen numeric field and (if a grouping field is available) compare group means.

You may need to install `matplotlib` and/or `seaborn` (`pip install matplotlib seaborn`) if not yet available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of numeric field and group means if available
if 'filtered_df' in locals() and not filtered_df.empty and 'numeric_field_id' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True, bins=10)
    plt.title(f'Distribution of Numeric Field ({numeric_field_id})')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping was done, show group mean barplot
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.xticks(rotation=45, ha='right')
        plt.title(f'Mean of {numeric_field_id} by {group_field_id}')
        plt.show()
else:
    print('No data to plot. Please ensure EDA section loaded and filtered data correctly.')

## 6. Conclusion

- We successfully loaded the **FAIR^2** dataset using the Croissant metadata standard and explored its available record sets and fields by their `@id`.
- After extracting tabular data, we performed initial filtering and normalization on a selected numeric field, and (where possible) grouped results by a categorical field.
- Data visualization gave insight into value distributions and group variations.
- This workflow can be repeated for any Croissant-compliant dataset: always reference fields, record sets, and columns by their unique `@id`, and use the `mlcroissant` API for data loading and inspection.

**Next steps:**
- Deeper domain-specific statistical analysis.
- Integrate additional FAIR datasets for broader studies.
- Explore advanced data curation and quality checks using `mlcroissant`.